# Встреча 1. Стоит ли тестировать и на чём измерять

Сквозной кейс — интернет-магазин. Все числа встречи — про него:

| Параметр | Значение |
| :--- | :--- |
| Конверсия в заказ на чекауте | 5% |
| Пользователей на чекауте | 50 000 / нед |
| Выручка | 3 000 000 ₽ / нед |
| Горизонт | 12 месяцев = 52 недели |
| Затронутая выручка за горизонт | 156 млн ₽ |

Допущение на всю встречу: относительный рост конверсии даёт такой же относительный рост выручки. Это разумно для чекаута, где корзина уже сформирована.

---

## Тема 1. Окупается ли тест

### Теория

**Зачем считать окупаемость.** В бизнесе корректно проведённый тест сам по себе ничего не стоит: ценность имеет только решение, которое меняет метрику на деньгах. Тест — это одновременно две вещи:

1. **Затраты** — фиксированная сумма, которую мы платим независимо от результата (разработка, инфраструктура, ожидание).
2. **Опция** — право внедрить вариант только тогда, когда данные говорят, что он окупится, и отказаться, когда нет.

Из этого сразу следует критерий из трёх действий, который мы считаем для каждой гипотезы:

- **Тестировать** — если ожидаемая прибыль теста (с учётом вероятности внедрить только хорошее) больше, чем у альтернатив.
- **Внедрять без теста** — если мы настолько уверены в эффекте, что задержка и затраты на тест не окупаются.
- **Не внедрять** — если даже лучшее из двух предыдущих действий в среднем убыточно.

**Точка безубыточности (break-even).** Интуиция: запуск варианта требует фиксированных денег (тест + внедрение + поддержка за весь горизонт). Какой минимальный эффект должен дать вариант, чтобы эти деньги отбить?

$$ \theta_{BE} = \frac{C_{test} + C_{impl} + C_{maint}}{R_{week} \cdot H} $$

где $R_{week}$ — недельная выручка, $H$ — горизонт в неделях. Значение $\theta_{BE}$ — относительный прирост выручки. Обычно это доли процента: затраты на тест малы по сравнению с выручкой, которую вариант будет приносить целый год.

- **Когда применять:** до любого теста, чтобы понять, что такое «успех» в деньгах, а не в p-value.
- **Типичная ошибка:** считать break-even по одним затратам на тест и забыть про внедрение и поддержку. Поддержка 20 тыс ₽/мес за год — это 240 тыс ₽, часто больше самого теста.

**MDE в деньгах.** Minimum Detectable Effect — минимальный эффект, который тест данного размера обнаруживает с заданной вероятностью. Для биномиальной метрики с базой $p$ и $n$ на группу:

$$ MDE_{rel} = \frac{(z_{1-\alpha/2} + z_{1-\beta})\sqrt{2p(1-p)/n}}{p} $$

В деньгах это $MDE_{rel} \cdot R_{week}$ — сколько ₽/нед мы в принципе способны «увидеть».

Ключевой смысл, который чаще всего путают: **MDE — это разрешающая способность теста, а не порог решения.** MDE отвечает на вопрос «какой эффект я заметлю с вероятностью 80%», а не «какой эффект мне выгоден». Порог решения — это $\theta_{BE}$. Тест может быть прибыльным, даже если его MDE много больше break-even: ценность теста создаёт не детектирование конкретной точки, а интегральный отбор хороших вариантов из всего распределения ожиданий (об этом — разбор H1 ниже).

- **Когда применять:** чтобы понять, не смешон ли размер выборки по сравнению с экономически значимой величиной.
- **Типичная ошибка:** сравнивать MDE с нулём («мы увидим любой эффект») и игнорировать, что MDE выражается в процентах от базы — при базе 5% MDE 7.72% относительных это 0.386 п.п. абсолютных.

**Сопоставление break-even и MDE.** Если $MDE \gg \theta_{BE}$, тест при данном $n$ не «разрешает» саму точку безубыточности — но это не приговор: решение принимается по порогу $\theta_{BE}$ с односторонним тестом, а выгода считается интегрально по всем эффектам. Если $MDE \ll \theta_{BE}$, тест легко различает выгодные и невыгодные варианты, и в нём почти нет смысла — дешевле внедрить без теста.

**ICE для очереди гипотез.** Грубая эвристика приоритизации, когда гипотез много, а руки одни: $ICE = Impact \times Confidence \times Ease$. Impact — ожидаемый масштаб эффекта, Confidence — уверенность в оценке (узкий/широкий разброс, история похожих тестов), Ease — обратная стоимость. ICE расставляет очередь, но финальное решение даёт только расчёт экономики ниже — ICE не видит ни горизонта, ни вероятности внедрения.

Три гипотезы встречи:

| | Ожидание | Тест | Внедрение | Поддержка | Откат |
| :--- | :--- | ---: | ---: | ---: | ---: |
| **H1** Купон отдельной строкой | 0% ± 3% | 180 тыс | 120 тыс | 20 тыс/мес | 250 тыс |
| **H2** Редизайн корзины | +3.3% ± 1.5% | 120 тыс | 80 тыс | 10 тыс/мес | 100 тыс |
| **H3** Новый платёжный провайдер | −1% ± 2% | 400 тыс | 700 тыс | 30 тыс/мес | 2 500 тыс |

### Практика

Сначала фиксируем кейс и гипотезы.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# --- Сквозной кейс: интернет-магазин ---
BASELINE_CR     = 0.05          # конверсия в заказ
WEEKLY_USERS    = 50_000        # пользователей/нед на чекауте
WEEKLY_REVENUE  = 3_000_000     # выручка, ₽/нед
HORIZON_MONTHS  = 12
HORIZON_WEEKS   = HORIZON_MONTHS * 52 / 12      # = 52 недели
REVENUE_HORIZON = WEEKLY_REVENUE * HORIZON_WEEKS

print(f"Горизонт {HORIZON_WEEKS:.0f} недель, затронутая выручка {REVENUE_HORIZON:,.0f} ₽")

# Гипотезы: μ и σ — среднее и разброс ожидаемого относительного эффекта.
HYP = {
    "H1 купон строкой":       dict(mu=0.000, sigma=0.030, test=180_000, impl=120_000, support=20_000, rollback=250_000),
    "H2 редизайн корзины":    dict(mu=0.033, sigma=0.015, test=120_000, impl=80_000,  support=10_000, rollback=100_000),
    "H3 платёжный провайдер": dict(mu=-0.010, sigma=0.020, test=400_000, impl=700_000, support=30_000, rollback=2_500_000),
}


Точка безубыточности: в затраты входят тест, внедрение и поддержка за весь горизонт (12 месяцев).

In [ ]:
def break_even(h):
    """Относительный эффект, при котором выручка за горизонт покрывает фиксированные затраты."""
    fixed = h["test"] + h["impl"] + h["support"] * HORIZON_MONTHS
    return fixed / REVENUE_HORIZON

for name, h in HYP.items():
    be = break_even(h)
    fixed = h["test"] + h["impl"] + h["support"] * HORIZON_MONTHS
    print(f"{name:24s} θ_BE = {be:.3%}   ({fixed:,.0f} ₽ всего; {be * WEEKLY_REVENUE:,.0f} ₽/нед)")


MDE при $n = 50\,000$ на группу — разрешающая способность стандартного дизайна.

In [ ]:
def mde_rel(n, alpha=0.05, power=0.80):
    z_a = stats.norm.ppf(1 - alpha / 2)   # двусторонний α
    z_b = stats.norm.ppf(power)
    se = np.sqrt(2 * BASELINE_CR * (1 - BASELINE_CR) / n)
    return (z_a + z_b) * se / BASELINE_CR

n_ref = 50_000
mde = mde_rel(n_ref)
print(f"MDE (n={n_ref:,}/группу): {mde:.2%} относительный = "
      f"{mde * BASELINE_CR * 100:.3f} п.п. = {mde * WEEKLY_REVENUE:,.0f} ₽/нед")


Теперь ключевой блок — модель «тест как опция». У каждого истинного эффекта $\theta$ есть своя **вероятность внедрения** (power против порога $\theta_{BE}$, односторонний $\alpha$). Выгоды и затраты взвешиваются по распределению ожиданий $\pi(\theta) = \mathcal{N}(\mu, \sigma)$ и по этой вероятности.

$$ POW(\theta) = \Phi\!\left(\frac{\delta_{true}(\theta) - \delta_{BE}}{SE} - z_{1-\alpha}\right) $$

$$ Cost_{RA} = C_{test} + \int \pi(\theta)\Big[\max(-\theta R_{test},0) + \big(\max(-\theta R_{post},0) + C_d(\theta)\big)POW(\theta)\Big] d\theta $$

$$ Gain_{RA} = \int \pi(\theta)\Big[\max(\theta R_{test},0) + \big(\max(\theta R_{post},0) + G_d\big)POW(\theta)\Big] d\theta $$

In [ ]:
effect_grid = np.linspace(-0.10, 0.10, 1000)

def adoption_prob(h, relative_effect, n, alpha):
    """P(внедрения | истинный эффект θ). Односторонняя гипотеза: θ > θ_BE."""
    p_c = BASELINE_CR
    p_t = np.clip(p_c * (1 + relative_effect), 1e-6, 1 - 1e-6)
    delta_true = p_t - p_c
    delta_be = p_c * break_even(h)
    se = np.sqrt(p_c * (1 - p_c) / n + p_t * (1 - p_t) / n)
    return stats.norm.sf(stats.norm.ppf(1 - alpha) - (delta_true - delta_be) / se)

def evaluate(h, n, alpha):
    density = stats.norm.pdf(effect_grid, h["mu"], h["sigma"])
    density /= np.trapezoid(density, effect_grid)   # сетка обрезает хвосты — нормируем
    pow_theta = adoption_prob(h, effect_grid, n, alpha)

    test_weeks = 2 * n / WEEKLY_USERS               # тест держит весь трафик
    post_weeks = max(HORIZON_WEEKS - test_weeks, 0) # и откладывает внедрение

    test_value = WEEKLY_REVENUE * effect_grid * test_weeks * 0.5   # во время теста — пол-трафика
    post_value = WEEKLY_REVENUE * effect_grid * post_weeks * pow_theta
    decision_cost = (h["impl"] + h["support"] * HORIZON_MONTHS
                     + h["rollback"] * (effect_grid < break_even(h))) * pow_theta

    gain = np.trapezoid(density * (np.maximum(test_value, 0) + np.maximum(post_value, 0)), effect_grid)
    loss = h["test"] + np.trapezoid(density * (np.maximum(-test_value, 0)
                                               + np.maximum(-post_value, 0) + decision_cost), effect_grid)
    p_adopt = np.trapezoid(density * pow_theta, effect_grid)
    return dict(gain=gain, loss=loss, net=gain - loss, rr=gain / loss, p_adopt=p_adopt)


Оценка всех трёх гипотез при $n = 50\,000$ и одностороннем $\alpha = 0.05$ (напомню: классический двусторонний 0.05 — это односторонний 0.025, то есть строже). Для вердикта считаем ещё «внедрить без теста»: $\mu \cdot R_{week}\cdot H - (C_{impl}+C_{maint})$.

In [ ]:
import pandas as pd

alpha_ref = 0.05   # односторонний
rows = []
for name, h in HYP.items():
    r = evaluate(h, n_ref, alpha_ref)
    ship = h["mu"] * REVENUE_HORIZON - (h["impl"] + h["support"] * HORIZON_MONTHS)
    rows.append((name, r["p_adopt"], r["gain"], r["loss"], r["net"], r["rr"], ship))

df = pd.DataFrame(rows, columns=["Гипотеза", "P(внедр.)", "Выгоды, ₽", "Затраты, ₽",
                                 "Чистая прибыль, ₽", "RR", "Внедрить без теста, ₽"]).round(0)
df


**Разбор «почему H1 выгодна тестировать».** У H1 ожидание эффекта ровно ноль: если внедрять без теста, половина вариантов окажется вредной, и средний итог — чистый убыток −360 тыс ₽ (платим внедрение и поддержку, а эффекта в среднем нет). Тест превращает это в +408 тыс ₽ прибыли, хотя вероятность внедрения всего 11.4%. Смотрим, из чего складывается эта прибыль:

In [ ]:
h1 = HYP["H1 купон строкой"]
r1 = evaluate(h1, n_ref, alpha_ref)

density = stats.norm.pdf(effect_grid, h1["mu"], h1["sigma"])
density /= np.trapezoid(density, effect_grid)
pow_theta = adoption_prob(h1, effect_grid, n_ref, alpha_ref)
tw = 2 * n_ref / WEEKLY_USERS
pw = HORIZON_WEEKS - tw
test_val = WEEKLY_REVENUE * effect_grid * tw * 0.5
post_val = WEEKLY_REVENUE * effect_grid * pw * pow_theta
dec_cost = (h1["impl"] + h1["support"] * HORIZON_MONTHS
            + h1["rollback"] * (effect_grid < break_even(h1))) * pow_theta

test_gain = np.trapezoid(density * np.maximum(test_val, 0), effect_grid)
post_gain = np.trapezoid(density * np.maximum(post_val, 0), effect_grid)
test_loss = np.trapezoid(density * np.maximum(-test_val, 0), effect_grid)
post_loss = np.trapezoid(density * np.maximum(-post_val, 0), effect_grid)
dec_loss  = np.trapezoid(density * dec_cost, effect_grid)

print(f"Выгода во время теста : {test_gain:>12,.0f} ₽")
print(f"Выгода после внедрения: {post_gain:>12,.0f} ₽")
print(f"Потеря во время теста : {test_loss:>12,.0f} ₽")
print(f"Потеря после внедрения: {post_loss:>12,.0f} ₽")
print(f"Ожидаемые затраты на внедрение/откат: {dec_loss:>12,.0f} ₽")
print(f"Фиксированные затраты на тест      : {h1['test']:>12,.0f} ₽")
print(f"ИТОГО чистая прибыль: {r1['net']:,.0f} ₽  при P(внедрения) = {r1['p_adopt']:.1%}")


Читаем декомпозицию. Во время теста мы почти ничего не зарабатываем и не теряем (+35.8 тыс / −35.8 тыс — симметрично, потому что эффект в среднем нулевой и трогаем только пол-трафика). Главный вклад — **выгода после внедрения: +637.6 тыс ₽**. Она взвешена вероятностью внедрения: мы внедряем только 11.4% вариантов, но именно те, где оценка эффекта оказалась сильно выше порога безубыточности. Тест работает как фильтр: отсекает ~88.6% «мусорных» исходов, оставляя хвост хороших. Вычитаем фиксированные 180 тыс на тест и ожидаемые ~42.8 тыс на внедрение/откат — остаётся +408 тыс ₽.

Вердикты по всем трём: **H1 — тестировать** (+408 тыс > −360 тыс > 0), **H2 — внедрять без теста** (+4.95 млн у внедрения против +1.88 млн у теста: задержка и неполная уверенность съедают больше, чем стоил бы тест), **H3 — не делать** (и тест, и внедрение в среднем убыточны из-за отрицательного ожидания).

---

## Тема 2. α и n как риск-менеджмент

### Теория

**Почему 0.05 — не догма.** $\alpha$ и $n$ — это не ритуал, а два рычага управления деньгами. $\alpha$ — цена, которую мы согласны платить за ложное внедрение; $n$ — сколько точности покупаем и как долго откладываем внедрение. Компромисс: маленький $\alpha$ даёт мало ложных внедрений, но медленно решает и теряет хорошие варианты; большой $\alpha$ решает быстро, но внедряет больше мусора. Оптимум зависит от цены ошибки (откат, необратимость) и горизонта: чем дешевле откат и длиннее горизонт, тем агрессивнее можно.

**Распределение ожиданий эффекта.** Истинный эффект $\theta$ до теста неизвестен, поэтому мы задаём априорное распределение $\pi(\theta) = \mathcal{N}(\mu,\sigma)$ по истории похожих тестов. Это не «байесовский вывод»: само решение остаётся частотным, $\pi(\theta)$ используется только для выбора $\alpha$ и $n$ на этапе дизайна. Опасность — в референтном классе: слишком узкая история даёт мало данных, слишком широкая — неточную оценку для конкретного случая.

**Выгоды и затраты с поправкой на риск.** Формулы из Тема 1: $Cost_{RA}$ и $Gain_{RA}$ интегрируют $\pi(\theta)$, помноженную на денежные исходы и вероятность внедрения $POW(\theta)$. Два итоговых числа отвечают на разные вопросы:

- **Чистая прибыль** $Net = Gain_{RA} - Cost_{RA}$ — сколько в среднем заработаем на этом решении. Смотрим, когда решение одно и важен абсолютный результат.
- **Отношение выгоды к риску** $RR = Gain_{RA} / Cost_{RA}$ — сколько ₽ выгоды на ₽ затрат. Смотрим, когда тестов много, а ресурсы ограничены и нужна лучшая отдача на вложенный рубль. $RR = 1$ означает $ROI = 0$.

Критерии расходятся: максимум прибыли обычно у более крупного и «мягкого» теста, максимум RR — у более экономного.

**Вероятность внедрения.** Решение привязывается не к нулю, а к точке безубыточности: внедряем, если есть основания считать, что эффект **выше** $\theta_{BE}$.

$$ H_0:\ \theta \le \theta_{BE}, \qquad H_1:\ \theta > \theta_{BE} $$

Внедряем при $\hat\delta > \delta_{BE} + z_{1-\alpha}\cdot SE$. Здесь $\alpha$ **односторонний** и равен вероятности внедрения при $\theta = \theta_{BE}$. Двусторонний 0.05 = односторонний 0.025. Заметьте асимметрию: вероятность внедрения растёт с эффектом, поэтому $\alpha = 0.5$ — это не монетка, а правило «внедряем, если оценка выше порога», и оно отбирает хорошие варианты чаще плохих.

**Ошибки Type S и Type M.** Даже значимый результат может врать. Для нормальной оценки всё определяется $\lambda = |\theta|/SE$ — отношением истинного эффекта к шуму:

$$ P_+ = 1-\Phi(z-\lambda),\qquad P_- = \Phi(-z-\lambda),\qquad Power = P_+ + P_- $$

$$ \text{Type S} = \frac{P_-}{P_+ + P_-}, \qquad \text{Type M} = \frac{\lambda(P_+ - P_-) + \varphi(z-\lambda) + \varphi(z+\lambda)}{\lambda(P_+ + P_-)} $$

- **Type S** — вероятность, что среди значимых результатов знак окажется неверным (фича «ухудшает», а на деле улучшает). Это вероятность, а не отношение.
- **Type M** — во сколько раз значимая оценка в среднем завышает истинный эффект по модулю (всегда ≥ 1).

Цена ошибок прямая: Type M заставляет закладывать в бизнес-план завышенный uplift (потом он не воспроизводится), Type S — принять решение в неправильную сторону. Обе ошибки растут, когда истинный эффект мал относительно шума, то есть ровно тогда, когда тест маломощный.

### Практика

Сетка $\alpha \in \{0.05, 0.1, 0.2, 0.3, 0.5\}$ × $n \in \{10\text{k} \dots 600\text{k}\}$ для H1. Ищем максимум прибыли и максимум RR.

In [ ]:
alphas = np.array([0.05, 0.1, 0.2, 0.3, 0.5])
ns = np.array([10_000, 25_000, 50_000, 100_000, 150_000, 200_000,
               300_000, 400_000, 500_000, 600_000])

net_grid = np.zeros((len(alphas), len(ns)))
rr_grid  = np.zeros((len(alphas), len(ns)))
for i, a in enumerate(alphas):
    for j, n in enumerate(ns):
        r = evaluate(h1, n, a)
        net_grid[i, j] = r["net"]
        rr_grid[i, j]  = r["rr"]

i_net = np.unravel_index(np.argmax(net_grid), net_grid.shape)
i_rr  = np.unravel_index(np.argmax(rr_grid),  rr_grid.shape)
print(f"Максимум прибыли: α={alphas[i_net[0]]}, n={ns[i_net[1]]:,} → "
      f"{net_grid[i_net]:,.0f} ₽ (RR {rr_grid[i_net]:.2f})")
print(f"Максимум RR:      α={alphas[i_rr[0]]}, n={ns[i_rr[1]]:,} → "
      f"RR {rr_grid[i_rr]:.2f} (прибыль {net_grid[i_rr]:,.0f} ₽)")


Тепловая карта: по горизонтали $n$, по вертикали $\alpha$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, values, title, fmt in [
    (axes[0], net_grid / 1e6, "Чистая прибыль, млн ₽", lambda v: f"{v:+.2f}"),
    (axes[1], rr_grid, "Отношение выгоды к риску (RR)", lambda v: f"{v:.2f}"),
]:
    im = ax.imshow(values, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(len(ns)), [f"{n//1000}k" for n in ns])
    ax.set_yticks(range(len(alphas)), alphas)
    ax.set_xlabel("n на группу"); ax.set_ylabel("α (односторонний)")
    ax.set_title(title)
    for i in range(len(alphas)):
        for j in range(len(ns)):
            ax.text(j, i, fmt(values[i, j]), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


Читаем оптимумы. **Прибыль** максимальна при $\alpha=0.5$, $n=150\,000$: +1.08 млн ₽. Здесь правило внедрения максимально мягкое («внедряем, если оценка выше порога безубыточности»), а 150 тыс на группу — это всего 6 недель теста: дальше задержка внедрения съедает больше, чем приносит точность. **RR** максимален при $\alpha=0.2$, $n=100\,000$: 3.44. Более строгий порог экономит затраты на внедрение/откат, а меньшая выборка дешевле, хотя абсолютная прибыль ниже (938 тыс против 1.08 млн). Это и есть расхождение критериев: если ресурс один — смотрим RR, если важен результат этого решения — прибыль.

Теперь цена малой мощности: Type S и Type M для маленьких истинных эффектов.

In [ ]:
def type_sm(p_t, n, alpha=0.05):
    p_c = BASELINE_CR
    theta = p_t - p_c
    se = np.sqrt(p_c * (1 - p_c) / n + p_t * (1 - p_t) / n)
    lam = abs(theta) / se
    z = stats.norm.ppf(1 - alpha / 2)
    P_plus  = 1 - stats.norm.cdf(z - lam)
    P_minus = stats.norm.cdf(-z - lam)
    power = P_plus + P_minus
    type_s = P_minus / power
    type_m = (lam * (P_plus - P_minus) + stats.norm.pdf(z - lam)
              + stats.norm.pdf(z + lam)) / (lam * power)
    return dict(theta=theta, se=se, lam=lam, power=power, type_s=type_s, type_m=type_m)

for pt in [0.0505, 0.0520]:   # +0.05 п.п. и +0.20 п.п.
    r = type_sm(pt, n_ref)
    print(f"истинный эффект {r['theta']*100:+.2f} п.п. (λ={r['lam']:.2f}): "
          f"мощность {r['power']:.1%}, Type S {r['type_s']:.2%}, Type M {r['type_m']:.2f}×")


Интерпретация. Истинный эффект +0.05 п.п. — это +1% относительный, почти в 8 раз меньше MDE (7.72%). Мощность всего 6.5%: тест почти никогда его не увидит, а когда увидит — среди «побед» 15.5% окажутся с неверным знаком, а модуль эффекта будет завышен в среднем в 6.55 раза. Если же истинный эффект +0.20 п.п. (+4%), мощность 30%, Type S почти нулевая (0.11%), Type M — 1.81×. Мораль: маломощный тест не просто «может не заметить» — он превращает редкие значимые результаты в систематически раздутые.

---

## Тема 3. Единица и метрика

### Теория

**Единица анализа = единица рандомизации.** Когда мы рандомизируем кластеры (продавца, пользователя, магазин), а метрику считаем на более мелких единицах (покупатель, сессия, чек), эти мелкие единицы внутри кластера **зависимы**. Запишем исход $i$-й единицы в кластере $j$:

$$ Y_{ij} = \mu + u_j + e_{ij}, \qquad u_j \sim \mathcal{N}(0,\sigma_u^2), \quad e_{ij} \sim \mathcal{N}(0,\sigma_e^2) $$

Доля дисперсии, приходящаяся на различия между кластерами, — **внутриклассовая корреляция (ICC)**:

$$ \rho = \frac{\sigma_u^2}{\sigma_u^2 + \sigma_e^2} $$

**Дизайн-эффект.** Среднее кластера размера $m$ имеет дисперсию в $DE$ раз больше, чем у $m$ независимых наблюдений:

$$ DE = 1 + (m-1)\rho, \qquad n_{eff} = \frac{N}{DE} $$

Интуиция: $m$ наблюдений одного кластера не дают $m$ независимых «голосов» — они частично дублируют друг друга. ICC 0.01 выглядит пренебрежимо, но умножается на $m$: при 1000 покупателей на продавца $DE \approx 11$.

- **Когда применять:** всякий раз, когда единица рандомизации крупнее единицы анализа.
- **Типичная ошибка:** рандомизировать кластеры, а тест считать по мелким единицам как по независимым. Наивная дисперсия занижается в $DE$ раз, ошибка I рода взлетает: при ICC 0.01 и 1000 на кластер ложных срабатываний ~55% вместо 5%. Кластерный MDE равен наивному, умноженному на $\sqrt{DE}$.

**Мощность определяет число кластеров, а не их размер.** При $m \to \infty$ кластерный MDE упирается в предел $\propto \sqrt{\rho / k}$, где $k$ — число кластеров на группу. Наращивать пользователей внутри кластера почти бесполезно, расти должно число кластеров.

**Метрика как решение.** Метрики различают по роли — что мы с ними делаем:

| Роль | Что это | Пример (магазин) |
| :--- | :--- | :--- |
| **Качество данных** | можно ли вообще верить тесту | SRM (доли групп), потеря телеметрии |
| **OEC** | метрика, по которой решаем «запускать или нет» | конверсия в заказ |
| **Guardrail** | не должна ухудшиться, даже если OEC растёт | средний чек, доля возвратов, время до заказа |
| **Диагностика** | объясняет, откуда взялось движение OEC | клики по кнопке, шаги воронки |

Порядок чтения результата следует ролям: сначала качество данных, потом OEC, потом guardrail, потом диагностика.

**Закон Гудхарта.** «Когда мера становится целью, она перестаёт быть хорошей мерой». Если OEC — единственная цель, её начинают двигать любыми способами, в том числе вредными (накрутка, ухудшение скрытых метрик). Защита — guardrail и контрметрики, которые ловят побочный ущерб.

**Ratio-метрика и её знаменатель.** Метрики вида «выручка/пользователь», «заказы/сессия» — отношения. У них два источника движения: числитель и знаменатель. Нельзя интерпретировать сдвиг отношения, не посмотрев на оба по отдельности: рост «заказов/сессия» может быть ростом заказов, а может — вымиранием дешёвых сессий. Знаменатель сам может реагировать на воздействие (metric SRM), и тогда отношение вводит в заблуждение.

### Практика

Три кейса, в каждом — единица рандомизации крупнее единицы анализа.

In [ ]:
cases = [
    # (название, кластеров/группу, наблюдений в кластере, ICC)
    ("Маркетплейс",  400,     150,  0.15),   # продавцы → покупатели
    ("Соцсеть",      100_000, 8,    0.20),   # пользователи → сессии
    ("Оффлайн-сеть", 40,      1200, 0.02),   # магазины → чеки
]

z_a = stats.norm.ppf(0.975)
z_b = stats.norm.ppf(0.8)

def mde_binom(n):   # иллюстративная конверсия 5% (тот же масштаб, что у магазина)
    return (z_a + z_b) * np.sqrt(2 * 0.05 * 0.95 / n) / 0.05

for name, k, m, rho in cases:
    de = 1 + (m - 1) * rho
    n_total = k * m
    n_eff = n_total / de
    naive = mde_binom(n_total)
    cluster = mde_binom(n_eff)
    print(f"{name:14s} DE = {de:6.2f} | N = {n_total:>9,} | N_eff = {n_eff:>9,.0f} | "
          f"наивный MDE {naive:.1%} → кластерный {cluster:.1%}")


Что здесь происходит.

- **Маркетплейс.** Рандомизируем 400 продавцов на группу, метрика — конверсия покупателя. 150 покупателей продавца сильно похожи (ICC 0.15), $DE = 23.35$: из 60 000 покупателей эффективных остаётся ~2 570. Наивный MDE 7.05% превращается в кластерный 34.07% — тест «слепнет» в 4.8 раза. Единица анализа — покупатель, но анализировать надо кластерно (cluster-robust SE или агрегация до продавца).
- **Соцсеть.** Рандомизируем 100 000 пользователей, метрика — на сессии. Сессии внутри пользователя коррелируют (ICC 0.20), но $m=8$ мало, поэтому $DE$ всего 2.40 — это терпимая цена. Наивный MDE 1.93% → кластерный 2.99%.
- **Оффлайн-сеть.** Рандомизируем 40 магазинов, метрика — на чек. ICC всего 0.02, но $m=1200$: $DE = 24.98$. Из 48 000 чеков эффективных ~1 922, MDE 7.88% → 39.40%. Кластеров мало (40 на группу), поэтому критерий — t-квантили с $2(k-1)=78$ степенями свободы, а не нормальные, и решающим рычагом здесь может быть только рост числа магазинов, а не чеков.

**Упражнение: выбери OEC и guardrail для H2.** H2 — редизайн корзины (+3.3% ± 1.5%). Предложи OEC, два guardrail и одну диагностическую метрику, и обоснуй, почему редизайн корзины может «накрутить» OEC в ущерб чему-то скрытому.

Рабочий вариант: **OEC** — конверсия в заказ (это то, ради чего редизайн). **Guardrail** — средний чек (редизайн может упростить корзину и потерять допродажи), доля возвратов/отмен (если корзина стала «продавать» агрессивнее), время до заказа (если новый интерфейс медленнее). **Диагностика** — клики по элементам новой корзины, доля убравших/добавивших товар. Ловушка Гудхарта: если оптимизировать только конверсию в заказ, легко съехать в «упрощаем до одной кнопки» и потерять средний чек — его и стережёт guardrail.

---

## Выводы

1. **Сначала экономика, потом статистика.** Порог решения — не p-value и не MDE, а точка безубыточности $\theta_{BE}$. MDE — только разрешающая способность теста.
2. **Тест — это опция.** Он ценен не «знанием», а отбором: H1 при нулевом ожидании и вероятности внедрения 11.4% превращает −360 тыс ₽ (внедрять вслепую) в +408 тыс ₽.
3. **Решения бывают разные:** H1 — тестировать, H2 — внедрять без теста, H3 — не делать. Один и тот же инструмент даёт три разных действия.
4. **$\alpha$ и $n$ подбираются под цену ошибки и горизонт.** Для H1 оптимум прибыли — $\alpha=0.5$, $n=150$k (+1.08 млн ₽), оптимум RR — $\alpha=0.2$, $n=100$k (3.44). 0.05 — частный случай, а не закон.
5. **Маломощный тест не просто «слеп» — он врёт.** Для истинного +0.05 п.п. значимые результаты завышены в 6.55 раза и в 15.5% случаев имеют неверный знак.
6. **Единица рандомизации и анализа должны совпадать** — или надо платить дизайн-эффектом. $DE = 1+(m-1)\rho$ превращает «100 000 пользователей» в горстку эффективных наблюдений.
7. **Метрика — это решение, а не число.** OEC решает «запускать ли», guardrail — «не сломали ли», диагностика — «почему». Гудхарт и знаменатель ratio-метрики — две главные ловушки интерпретации.